# 09 — Retries, Rate Limits, and Idempotency

In [ ]:
# Setup: run this first. On Google Colab (or anywhere the course files are missing) it
# downloads the course and moves into this notebook's folder. Elsewhere it does nothing.
import os, sys
if not os.path.exists("../README.md"):
    course = os.path.join(os.path.expanduser("~"), "python-notebook")
    if not os.path.isdir(course):
        os.system(f"git clone -q --depth 1 https://github.com/SwarmLadder/python-notebook {course}")
    os.chdir(os.path.join(course, "advanced", "notebooks"))
    sys.path.insert(0, os.getcwd())

In [ ]:
from lesson import Exercises, ___, run

check = Exercises('advanced/09', {1: 'd14f6c107508fefa', 2: '73977cb50902cb75', 3: 'b2319b3a0085464a', 4: '58024fe14bedfcf8', 5: '5e6208c4499fadc3', 6: '4877c3cede5da603', 7: '46ea29b80671d463', 8: '6da26c48ebca8f00', 9: '00203a04302399ee'}, practice='advanced_02_concurrency_resilience')

## 1. Classify, then retry

- Only retry failures that may fix themselves (timeouts, 503s). Re-raise permanent ones (bad input, 404) at once.
- Use separate exception classes: `TransientError` and `PermanentError`.
- Cap the attempts and re-raise the last error when they run out.

In [ ]:
class TransientError(Exception): pass
class PermanentError(Exception): pass

calls = []
def flaky():
    calls.append(1)
    if len(calls) < 3:
        raise TransientError("try again")
    return "ok"

for attempt in range(5):
    try:
        print(flaky(), "after", attempt + 1, "attempts")
        break
    except TransientError:
        continue

**Exercise 1.** Which of these are worth retrying? Keep the transient status codes.

In [ ]:
statuses = [400, 404, 429, 500, 503]
retryable = [s for s in statuses if s == 429 or s >= ___]
check(1, retryable)

**Exercise 2.** An operation fails twice with a transient error, then succeeds. With 3 attempts, how many calls are made?

In [ ]:
predicted = ___
check(2, predicted)

**Exercise 3.** A `PermanentError` on the first call: how many calls before it's raised?

In [ ]:
predicted = ___
check(3, predicted)

## 2. Backoff and jitter

- Wait longer each time: `base * 2 ** attempt` gives 0.1, 0.2, 0.4, …
- Add jitter so clients don't retry in lockstep: full jitter picks `random.uniform(0, base * 2 ** n)`.
- Inject `sleep` and the random generator (`random.Random(seed)`) so tests are instant and repeatable.

In [ ]:
import random
base = 0.1
print([base * 2 ** n for n in range(4)])
rng = random.Random(42)
print([round(rng.uniform(0, base * 2 ** n), 3) for n in range(4)])

**Exercise 4.** Exponential delays for 4 attempts with base 0.5.

In [ ]:
delays = [0.5 * 2 ** n for n in range(___)]
check(4, delays)

**Exercise 5.** Same seed, same delays? Compare two generators seeded with 7.

In [ ]:
import random
a = [random.Random(7).random() for _ in range(1)]
b = [random.Random(7).random() for _ in range(1)]
predicted = ___   # a == b
check(5, predicted)

**Exercise 6.** Record the waits with an injected `sleep` instead of really waiting.

In [ ]:
waits = []
fake_sleep = ___   # the list method that adds one item to waits
for attempt in range(3):
    fake_sleep(0.1 * 2 ** attempt)
check(6, [round(w, 2) for w in waits])

## 3. Idempotency, rate limits, circuit breakers

- An idempotency key makes a repeated request safe: run the side effect once and return the stored result for duplicates.
- A token bucket allows bursts up to `capacity` and refills at `rate` per second.
- A circuit breaker stops calling a failing service after `threshold` failures, and tries again after a cool-down.

In [ ]:
results = {}
charges = []
def charge(key, amount):
    if key in results:
        return results[key]
    charges.append(amount)
    results[key] = f"receipt-{len(charges)}"
    return results[key]

print(charge("k1", 10), charge("k1", 10), charge("k2", 5), charges)

**Exercise 7.** Charge with the same key twice. How many real charges happen?

In [ ]:
charges, results = [], {}
def charge(key, amount):
    if key not in results:
        charges.append(amount)
        results[key] = len(charges)
    return results[key]

charge("a", 5); charge("a", 5); charge("b", 7)
predicted = ___   # len(charges)
check(7, predicted)

**Exercise 8.** A bucket with 3 tokens refilled at 1 per second. After 2 seconds with 0 tokens, how many are there?

In [ ]:
tokens = min(3, 0 + 1 * ___)
check(8, tokens)

**Exercise 9.** A breaker with threshold 3: after 3 straight failures, is it open?

In [ ]:
failures, threshold = 0, 3
for _ in range(3):
    failures += 1
is_open = failures >= ___
check(9, is_open)

## Challenges

Real functions, graded by tests. Complete each implementation and run its check. The independent build supplies only the interface.

### 1. `TransientError`

A failure worth retrying.

In [ ]:
class TransientError(Exception):
    """A failure worth retrying (timeouts, 503s...)."""

In [ ]:
check('TransientError')

### 2. `PermanentError`

A failure that won't fix itself.

In [ ]:
class PermanentError(Exception):
    """A failure that will not fix itself (bad input, 404...)."""

In [ ]:
check('PermanentError')

### 3. `retry`

Retry transient errors with exponential backoff; re-raise permanent ones.

In [ ]:
def retry(operation: Callable[[], object], attempts: int = 3, base_delay: float = 0.1,
          sleep: Callable[[float], None] = None) -> object:
    """Call operation(); on TransientError wait base_delay * 2**n and try again.

    PermanentError is re-raised immediately. After `attempts` failures re-raise the
    last TransientError. `sleep` is injected so tests never really wait.
    """
    import time
    sleep = sleep or time.sleep
    for attempt in ___:
        try:
            return ___
        except TransientError:
            if ___:
                raise
            sleep(base_delay * 2 ** attempt)
    raise ValueError("attempts must be at least 1")

In [ ]:
check('retry')

### 4. `jittered_delays`

Full-jitter backoff delays; the same seed gives the same delays.

In [ ]:
def jittered_delays(attempts: int, base: float, seed: int) -> list[float]:
    """Full-jitter backoff: delay n is uniform in [0, base * 2**n]. Same seed -> same delays."""
    rng = random.Random(seed)
    return ___

In [ ]:
check('jittered_delays')

### Independent build — `IdempotentPayments`

Order Desk accepts IdempotentPayments(charge_card). charge(key, amount) calls charge_card(amount) and returns its receipt. Replaying a successful charge with the same key and amount returns that receipt without a second charge. A conflicting amount raises ValueError.

Implement the interface below, run its check, and explain one boundary case with a test of your own.

In [ ]:
class IdempotentPayments:
    pass  # Implement the public interface in the ticket.

In [ ]:
check('IdempotentPayments')

### 6. `TokenBucket`

A rate limiter refilled from an injected clock.

In [ ]:
class TokenBucket:
    """Rate limiter: `capacity` tokens, refilled at `rate` tokens/second from an injected clock.

    allow() consumes a token and returns True, or returns False if none are available.
    The bucket starts full and never holds more than capacity.
    """

    def __init__(self, capacity: int, rate: float, clock: Callable[[], float]):
        self.capacity, self.rate, self.clock = capacity, rate, clock
        self._tokens = ___
        self._last = clock()

    def allow(self) -> bool:
        now = ___
        self._tokens = min(self.capacity, self._tokens + (now - self._last) * self.rate)
        self._last = now
        if ___:
            self._tokens -= 1
            return True
        return False

In [ ]:
check('TokenBucket')

### 7. `CircuitOpen`

Raised instead of calling while the breaker is open.

In [ ]:
class CircuitOpen(Exception):
    """Raised instead of calling the operation while the breaker is open."""

In [ ]:
check('CircuitOpen')

### 8. `CircuitBreaker`

Open after `threshold` failures; try again after the cool-down.

In [ ]:
class CircuitBreaker:
    """After `threshold` consecutive failures the breaker opens: calls raise CircuitOpen
    without running. After `reset_after` seconds (injected clock) one trial call is allowed;
    success closes the breaker, failure re-opens it. Success resets the failure count.

    state is 'closed', 'open' or 'half-open'.
    """

    def __init__(self, threshold: int, reset_after: float, clock: Callable[[], float]):
        self.threshold, self.reset_after, self.clock = ___
        self.failures = 0
        self._opened_at = None

    @property
    def state(self) -> str:
        if ___:
            return "closed"
        if ___:
            return "half-open"
        return "open"

    def call(self, operation: Callable[[], object]):
        if ___:
            raise CircuitOpen("circuit is open")
        try:
            result = ___
        except Exception:
            self.failures += 1
            if ___:
                self._opened_at = self.clock()
            raise
        self.failures = 0
        self._opened_at = None
        return result

In [ ]:
check('CircuitBreaker')

## Quick quiz

Replace each `'?'` in the cell below with your choice and run it. The answers are hidden, but each one is checked.

**1.** Which error is worth retrying?

   - **a)** 404 Not Found
   - **b)** 400 Bad Request
   - **c)** 503 Service Unavailable / timeout

**2.** Why add jitter to backoff?

   - **a)** So many clients don't retry at the same instant
   - **b)** To make retries slower
   - **c)** Randomness is required

**3.** An idempotency key ensures…

   - **a)** Encryption
   - **b)** A repeated request has the same effect as one
   - **c)** Faster requests

In [ ]:
from quiz import Quiz

quiz = Quiz('advanced/09', {1: 'a61182c344df', 2: 'c2a9f61f0fa0', 3: 'b9dadf04c824'})
quiz.answer(1, '?')
quiz.answer(2, '?')
quiz.answer(3, '?')

**Next:** [10 — Profiling, Memory, and Honest Benchmarking](10_profiling_memory_benchmarks.ipynb)